# Imports

In [ ]:
#running on google colab
on_google_colab = False

In [ ]:
#google colab installs
if on_google_colab:
    !pip3 install pydicom

In [ ]:
#imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

import pydicom
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as TF
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [ ]:
#seed used for random
seed = 42
torch.manual_seed(seed)

In [ ]:
#device
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

In [ ]:
#colab settings
if on_google_colab:
    from google.colab import drive
    drive.mount('/content/drive')
    os.chdir('/content/drive/MyDrive/upenn_measurement_project/model_directory')

# Measurement data

In [ ]:
#read csv
measurement_file = "08172025.csv"

df = pd.read_csv(f"../data/measurements/{measurement_file}")

print(df.columns)

#drop unnecessary columns
df.drop(columns = ["Student Name", "Notes", "Date Completed", "Unnamed: 24", "Unnamed: 25", "Unnamed: 26", "Unnamed: 27"], inplace = True)

print(df.columns)

print(len(df.columns.values))

In [ ]:
#get rid of images with na (only temporary for convenience)
df = df.dropna()

# Load image data (augmentation, datasets, and dataloaders)

In [ ]:
#image directory
img_dir = "../data/box_images"

#train test split
train_test_split = 0.8
val_test_split = 0.5

#batch size
train_batch_size = 32
val_batch_size = 8
test_batch_size = 8

#pixels per mm conversion (known)
pix_per_mm = 2400 / 408

#training scale factor
img_scale_factor = 0.1

#image size (pix)
img_width = int(2400 * img_scale_factor)
img_height = int(1920 * img_scale_factor)

In [ ]:
# #data augmentation settings
# ##rotation
# rot_int = 10
# rot_st = 0
# rot_en = 0

# ##down scale
# scale_int = 0.125
# scale_st = 1
# scale_en = 1

# ##crop image
# crop_int = 0.2
# crop_st = 1
# crop_en = 1

# ##random background color
# rand_bg = True

# ##noise location (how much of the image get noised)
# noise_loc_int = 0.125
# noise_loc_st = 0
# noise_loc_en = 0.5

# ##noise level (how much noise)
# noise_level_int = 0.25
# noise_level_st = 0
# noise_level_en = 0.75

In [ ]:
#FIXME TESTING NO AUGMENTATION
##rotation
rot_int = 10
rot_st = 0
rot_en = 0

##down scale
scale_int = 0.125
scale_st = 1
scale_en = 1 #TODO can change this, 0.6 was chosen so no matter what rotation, no part of image is cut off

##crop image
crop_int = 0.2
crop_st = 1
crop_en = 1

##random background color
rand_bg = True

##noise location (how much of the image get noised)
noise_loc_int = 0.125
noise_loc_st = 0
noise_loc_en = 0

##noise level (how much noise)
noise_level_int = 0.25
noise_level_st = 0
noise_level_en = 0

In [ ]:
#class for each data point: image path, flip (if asking for left side of image), rotation, scale, true values for measurements
class DataPoint:
    def __init__(self, img_dir, img_file, img_flip, df, img_width, img_height, aug_rot = 0, aug_scale = 1, aug_crop = 1, aug_rand_bg = False, aug_noise_loc = 0, aug_noise_level = 0):
        self.img_path = img_dir + '/' + img_file
        self.img_flip = img_flip
        self.img_width = img_width
        self.img_height = img_height
        self.aug_rot = aug_rot
        self.aug_scale = aug_scale
        self.aug_crop = aug_crop
        self.aug_rand_bg_val = np.random.rand() if aug_rand_bg else 0
        self.aug_noise_loc = aug_noise_loc
        self.aug_noise_level = aug_noise_level
        self.y = torch.from_numpy(df.loc[df['ID'] == img_file].drop(columns = 'ID').values.astype(np.float32)).reshape(-1)

        #additionally required aug_scale given aug_rot
        abscos = np.abs(np.cos(self.aug_rot * np.pi / 180))
        abssin = np.abs(np.sin(self.aug_rot * np.pi / 180))
        self.aug_scale *= min(1 / (abscos + self.img_height / self.img_width * abssin), 1 / (self.img_width / self.img_height * abssin + abscos))

    #apply augmentation, return augmented image and rescaled values for measurements
    def get_image(self):
        ds = pydicom.dcmread(self.img_path)
        img = ds.pixel_array

        img = torch.from_numpy(img).float()
        img = (img - img.min()) / (img.max() - img.min() + 1e-5)
        img = img.unsqueeze(0).unsqueeze(0)

        #flip image if needed
        if self.img_flip:
            img = TF.hflip(img)

        #adding noise
        _, _, orig_height, orig_width = img.shape
        noise = torch.randn_like(img) * self.aug_noise_level
        mask = torch.ones_like(img)
        mask[:, :, int(orig_height * self.aug_noise_loc):, int(orig_width * self.aug_noise_loc):] = 0
        img = torch.clamp(img + noise * mask, 0, 1)

        #crop image
        _, _, orig_height, orig_width = img.shape
        img = img[:, :, int(orig_height * (1 - self.aug_crop)):, int(orig_width * (1 - self.aug_crop)):]

        #downscale image
        new_height = int(self.img_height * self.aug_scale)
        new_width = int(self.img_width * self.aug_scale)
        img = F.interpolate(img, size = (new_height, new_width), mode = 'bilinear', align_corners = False)

        #pad back to target size
        pad_height = self.img_height - new_height
        pad_width = self.img_width - new_width
        pad_u = pad_height // 2
        pad_d = pad_height - pad_u
        pad_l = pad_width // 2
        pad_r = pad_width - pad_l
        img = F.pad(img, (pad_l, pad_r, pad_u, pad_d), mode = 'constant', value = self.aug_rand_bg_val)

        #rotate image
        img = TF.rotate(img, angle = self.aug_rot, interpolation = TF.InterpolationMode.BILINEAR, fill = self.aug_rand_bg_val)

        #get measurements for correct side, rescale all measurements so they are in terms of pixels (except for angle measurement)
        y_aug = self.y.clone()
        if self.img_flip: #left side of image, right set of measurements
            y_aug = y_aug[10:]
        else: #right side of image, left set of measurements
            y_aug = y_aug[:10]
        y_aug[:9] *= self.aug_scale / self.aug_crop

        # img = torch.cat([img, custom_edge_filter(img)], dim = 1)
        # img = torch.cat([img, canny_edges(img), sobel_edge_manual(img)], dim = 1)
        # img = torch.cat([img, sobel_edge_manual(img)], dim = 1)
        # img = torch.cat([img, hough_corners(img)], dim = 1)
        return img.squeeze(0), y_aug, torch.tensor(self.aug_scale).float()


In [ ]:
class ImageDataset(Dataset):
    def __init__(self, data_points):
        self.data_points = data_points

    def __len__(self):
        return len(self.data_points)

    def __getitem__(self, idx):
        return self.data_points[idx].get_image()

In [ ]:
#all images in directory
all_img_in_dir = [i for i in os.listdir(img_dir) if i.lower().endswith("dcm")]

#check all images in df are loaded (else throw error)
for i in df['ID'].values:
    if i not in all_img_in_dir:
        raise SystemError(f'Directory is missing images: {i}')

#load only images in the df
all_img = [i for i in all_img_in_dir if i in df['ID'].values]
print(f'Images in use: {len(all_img)}/{len(all_img_in_dir)}')

#train and val sizes
train_sz = int(train_test_split * len(all_img))
val_sz = int(val_test_split * (len(all_img) - train_sz))

#split train, val, test sets
all_idx = torch.randperm(len(all_img))
train_img = [all_img[i] for i in all_idx[:train_sz]]
val_img = [all_img[i] for i in all_idx[train_sz:train_sz + val_sz]]
test_img = [all_img[i] for i in all_idx[train_sz + val_sz:]]

In [ ]:
#augmented training data in DataPoint class
train_data_points = []
for img_file in train_img:
    for img_flip in range(2):
        for aug_rot in np.arange(rot_st, rot_en + 1e-8, rot_int):
            for aug_scale in np.arange(scale_st, scale_en + 1e-8, scale_int):
                for aug_crop in np.arange(crop_st, crop_en + 1e-8, crop_int):
                    for aug_noise_loc in np.arange(noise_loc_st, noise_loc_en + 1e-8, noise_loc_int):
                        for aug_noise_level in np.arange(noise_level_st, noise_level_en + 1e-8, noise_level_int):
                            train_data_points.append(DataPoint(img_dir, img_file, img_flip, df, img_width, img_height, aug_rot, aug_scale, aug_crop, rand_bg, aug_noise_loc, aug_noise_level))
print(f'Augmented training set size: {len(train_data_points)}')

#validation data in DataPoint class (no augmentation transformations applied)
val_data_points = []
for img_file in val_img:
    for img_flip in range(2):
        val_data_points.append(DataPoint(img_dir, img_file, img_flip, df, img_width, img_height, 0, 1, 1, False)) #FIXME
print(f'Validation set size: {len(val_data_points)}')

#testing data in DataPoint class (no augmentation transformations applied)
test_data_points = []
for img_file in test_img:
    for img_flip in range(2):
        test_data_points.append(DataPoint(img_dir, img_file, img_flip, df, img_width, img_height, 0, 1, 1, False)) #FIXME
print(f'Testing set size: {len(test_data_points)}')

In [ ]:
#training dataset and data loader
train_set = ImageDataset(train_data_points)
train_loader = DataLoader(train_set, batch_size = train_batch_size, shuffle = True)

#validation dataset and data loader
val_set = ImageDataset(val_data_points)
val_loader = DataLoader(val_set, batch_size = val_batch_size, shuffle = False)

#testing dataset and data loader
test_set = ImageDataset(test_data_points)
test_loader = DataLoader(test_set, batch_size = test_batch_size, shuffle = False)

In [ ]:
#quick test
testcnt = 1
testchannelcnt = 1
for imgs, ys, aug_scales in train_loader:
    # print(imgs[0].shape)
    # print(ys.shape)
    # print(ys[0])
    # print(aug_scales.shape)
    # print(aug_scales[0])

    for i in range(testchannelcnt):
        plt.imshow(imgs[0][i], cmap = 'gray')
        plt.axis('off')
        plt.show()

    testcnt -= 1
    if testcnt == 0:
        break


# Unsupervised keypoint detection model (autoencoder)

## Architecture

In [ ]:
#heatmaps (reduced H, W) to keypoints (normalized)
def heatmaps_to_keypoints(heatmaps):
    device = heatmaps.device

    B, K, H, W = heatmaps.shape
    heatmaps = heatmaps.view(B, K, -1)
    softmax_heatmaps = F.softmax(heatmaps, dim=-1)
    softmax_heatmaps = softmax_heatmaps.view(B, K, H, W)

    xs = torch.linspace(-1, 1, W, device = device)
    ys = torch.linspace(-1, 1, H, device = device)
    grid_y, grid_x = torch.meshgrid(ys, xs, indexing = 'ij')
    grid_x = grid_x.unsqueeze(0).unsqueeze(0)
    grid_y = grid_y.unsqueeze(0).unsqueeze(0)

    expected_x = (softmax_heatmaps * grid_x).sum(dim= [-2, -1])
    expected_y = (softmax_heatmaps * grid_y).sum(dim= [-2, -1])

    return torch.stack([expected_x, expected_y], dim = 2)

In [ ]:
#keypoints (normalized) to gaussian (normalized)
def keypoints_to_gaussian(keypoints, height, width, sigma = 0.1):
    device = keypoints.device

    B, K, _ = keypoints.shape

    xs = torch.linspace(-1, 1, width, device = device)
    ys = torch.linspace(-1, 1, height, device = device)
    grid_y, grid_x = torch.meshgrid(ys, xs, indexing = 'ij')
    grid = torch.stack([grid_x, grid_y], dim = -1).view(1, 1, height, width, 2)

    keypoints = keypoints.view(B, K, 1, 1, 2)
    sq_dist = torch.sum((grid - keypoints) ** 2, dim = -1)

    return torch.exp(-sq_dist / (2 * sigma ** 2))


In [ ]:
#get keypoints in original pixel space from normalized keypoints
def get_keypoints(keypoints, img_height, img_width):
    keypoints = keypoints.clone()
    keypoints = (keypoints + 1) / 2
    keypoints[:, :, 0] = keypoints[:, :, 0] * img_width
    keypoints[:, :, 1] = keypoints[:, :, 1] * img_height
    return keypoints

In [ ]:
#keypoint detection encoder
class KeypointDetectionEncoder(nn.Module):
    def __init__(self, keypoint_cnt, img_height, img_width, stand_alone = True, channel_cnt = [1, 32, 64, 128, 256]):
        super().__init__()

        self.img_height = img_height
        self.img_width = img_width
        self.stand_alone = stand_alone

        self.conv1 = nn.Sequential(
            nn.Conv2d(channel_cnt[0], channel_cnt[1], kernel_size = 7, stride = 1, padding = 3), #192, 240
            nn.ReLU(),
            nn.Conv2d(channel_cnt[1], channel_cnt[1], kernel_size = 3, stride = 1, padding = 1), #192, 240
            nn.ReLU()
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(channel_cnt[1], channel_cnt[2], kernel_size = 3, stride = 2, padding = 1), #96, 120
            nn.ReLU(),
            nn.Conv2d(channel_cnt[2], channel_cnt[2], kernel_size = 3, stride = 1, padding = 1), #96, 120
            nn.ReLU()
        )

        self.conv3 = nn.Sequential(
            nn.Conv2d(channel_cnt[2], channel_cnt[3], kernel_size = 3, stride = 2, padding = 1), #48, 60
            nn.ReLU(),
            nn.Conv2d(channel_cnt[3], channel_cnt[3], kernel_size = 3, stride = 1, padding = 1), #48, 60
            nn.ReLU()
        )

        self.conv4 = nn.Sequential(
            nn.Conv2d(channel_cnt[3], channel_cnt[4], kernel_size = 3, stride = 2, padding = 1), #24, 30
            nn.ReLU(),
            nn.Conv2d(channel_cnt[4], channel_cnt[4], kernel_size = 3, stride = 1, padding = 1), #24, 30
            nn.ReLU()
        )

        self.conv5 = nn.Conv2d(channel_cnt[4], keypoint_cnt, kernel_size = 1, stride = 1, padding = 0)

    def forward(self, x):
        skips = []

        x = self.conv1(x)
        skips.append(x) #C1, 192, 240

        x = self.conv2(x)
        skips.append(x) #C2, 96, 120

        x = self.conv3(x)
        skips.append(x) #C3, 48, 60

        x = self.conv4(x)
        skips.append(x) #C4, 24, 30

        x = self.conv5(x)

        if self.stand_alone: #stand alone encoder
            x = heatmaps_to_keypoints(x)
            x = get_keypoints(x, self.img_height, self.img_width)
            return x

        return x, skips

In [ ]:
#keypoint detection decoder
class KeypointDetectionDecoder(nn.Module):
    def __init__(self, keypoint_cnt, channel_cnt):
        super().__init__()

        self.conv1 = nn.Sequential(
            nn.Conv2d(keypoint_cnt + channel_cnt[4], channel_cnt[4], kernel_size = 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.Conv2d(channel_cnt[4], channel_cnt[4], kernel_size = 3, stride = 1, padding = 1),
            nn.ReLU(),
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(channel_cnt[4], channel_cnt[3], kernel_size = 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.Conv2d(channel_cnt[3], channel_cnt[3], kernel_size = 3, stride = 1, padding = 1),
            nn.ReLU()
        )

        self.conv3 = nn.Sequential(
            nn.Conv2d(channel_cnt[3], channel_cnt[2], kernel_size = 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.Conv2d(channel_cnt[2], channel_cnt[2], kernel_size = 3, stride = 1, padding = 1),
            nn.ReLU()
        )

        self.conv4 = nn.Sequential(
            nn.Conv2d(channel_cnt[2], channel_cnt[1], kernel_size = 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.Conv2d(channel_cnt[1], channel_cnt[1], kernel_size = 3, stride = 1, padding = 1),
            nn.ReLU()
        )

        self.conv5 = nn.Sequential(
            nn.Conv2d(channel_cnt[1], channel_cnt[0], kernel_size = 3, stride = 1, padding = 1),
            nn.Sigmoid()
        )

    def forward(self, x, skips):
        x = torch.cat([x, skips[3]], dim = 1)

        x = self.conv1(x)

        x = F.interpolate(x, scale_factor = 2, mode = 'bilinear', align_corners = False)
        x = self.conv2(x)

        x = F.interpolate(x, scale_factor = 2, mode = 'bilinear', align_corners = False)
        x = self.conv3(x)

        x = F.interpolate(x, scale_factor = 2, mode = 'bilinear', align_corners = False)
        x = self.conv4(x)

        x = self.conv5(x)

        return x

In [ ]:
#full model
class KeypointDetectionModel(nn.Module):
    def __init__(self, keypoint_cnt, img_height, img_width, channel_cnt = [1, 32, 64, 128, 256]):
        super().__init__()

        self.img_height = img_height
        self.img_width = img_width

        self.encoder = KeypointDetectionEncoder(keypoint_cnt, self.img_height, self.img_width, False, channel_cnt)
        self.decoder = KeypointDetectionDecoder(keypoint_cnt, channel_cnt)

    def forward(self, source_img, tar_img):
        src_heatmaps, skips = self.encoder(source_img)
        tar_heatmaps, _ = self.encoder(tar_img)

        src_keypoints = heatmaps_to_keypoints(src_heatmaps)
        tar_keypoints = heatmaps_to_keypoints(tar_heatmaps)

        gaussians = keypoints_to_gaussian(tar_keypoints, tar_heatmaps.shape[-2], tar_heatmaps.shape[-1])

        recon = self.decoder(gaussians, skips)

        src_keypoints = get_keypoints(src_keypoints, self.img_height, self.img_width)
        tar_keypoints = get_keypoints(tar_keypoints, self.img_height, self.img_width)

        return recon, src_keypoints, tar_keypoints

In [ ]:
def untransform_keypoints(keypoints, img_height, img_width, adj_rot, adj_scale, adj_x, adj_y):
    keypoints = keypoints.clone()

    #center
    cx = img_width / 2
    cy = img_height / 2

    #center
    keypoints[:, :, 0] = keypoints[:, :, 0] - cx
    keypoints[:, :, 1] = keypoints[:, :, 1] - cy

    #offset
    keypoints[:, :, 0] = keypoints[:, :, 0] + adj_x
    keypoints[:, :, 1] = keypoints[:, :, 1] + adj_y

    #rotate
    adj_rot = adj_rot * torch.pi / 180
    cos_adj_rot = torch.cos(adj_rot)
    sin_adj_rot = torch.sin(adj_rot)
    adj_rot_mat = torch.stack([torch.stack([cos_adj_rot, -sin_adj_rot], dim = -1), torch.stack([sin_adj_rot, cos_adj_rot], dim = -1)], dim = -1)
    adj_rot_mat = adj_rot_mat.expand(-1, keypoints.shape[1], -1, -1)
    keypoints = adj_rot_mat @ keypoints.unsqueeze(-1)
    keypoints = keypoints.squeeze(-1)

    #scale
    keypoints = keypoints * adj_scale.view(-1, 1, 1)

    #uncenter
    keypoints[:, :, 0] = keypoints[:, :, 0] + cx
    keypoints[:, :, 1] = keypoints[:, :, 1] + cy

    return keypoints
        

## Data

In [ ]:
#batch sizes
keypoint_detection_train_batch_size = 256
keypoint_detection_val_batch_size = 64
keypoint_detection_test_batch_size = 64

In [ ]:
##source rotation
keypoint_detection_srcrot_int = 30
keypoint_detection_srcrot_st = 0
keypoint_detection_srcrot_en = 0

##target rotation
keypoint_detection_tarrot_int = 30
keypoint_detection_tarrot_st = -30
keypoint_detection_tarrot_en = 30

##target cropping
keypoint_detection_tarcrop_int = 0.2
keypoint_detection_tarcrop_st = 0.4
keypoint_detection_tarcrop_en = 0.4

##source x offset
keypoint_detection_srcx_int = 50
keypoint_detection_srcx_st = 0
keypoint_detection_srcx_en = 0

##source y offset
keypoint_detection_srcy_int = 50
keypoint_detection_srcy_st = 0
keypoint_detection_srcy_en = 0

##target x offset
keypoint_detection_tarx_int = 50
keypoint_detection_tarx_st = -50
keypoint_detection_tarx_en = 0

##target y offset
keypoint_detection_tary_int = 50
keypoint_detection_tary_st = -50
keypoint_detection_tary_en = 0

In [ ]:
#class for each data point used for keypoint detection
class KeypointDetectionDataPoint:
    def __init__(self, img_dir, img_file, img_flip, img_width, img_height, srcrot = 0, tarrot = 0, tarcrop = 0,srcx = 0, srcy = 0, tarx = 0, tary = 0):
        self.img_path = img_dir + '/' + img_file
        self.img_flip = img_flip
        self.img_width = img_width
        self.img_height = img_height

        self.srcrot = srcrot
        self.tarrot = srcrot + tarrot

        self.tarcrop = tarcrop

        self.srcx = srcx
        self.srcy = srcy
        self.tarx = tarx
        self.tary = tary

        self.srcscale = 1
        self.tarscale = 1

        #additionally requried scale for source img rotation
        abscos = np.abs(np.cos(self.srcrot * np.pi / 180))
        abssin = np.abs(np.sin(self.srcrot * np.pi / 180))
        self.srcscale *= min(1 / (abscos + self.img_height / self.img_width * abssin), 1 / (self.img_width / self.img_height * abssin + abscos))

        #additionally requried scale for target img rotation
        abscos = np.abs(np.cos(self.tarrot * np.pi / 180))
        abssin = np.abs(np.sin(self.tarrot * np.pi / 180))
        self.tarscale *= min(1 / (abscos + self.img_height / self.img_width * abssin), 1 / (self.img_width / self.img_height * abssin + abscos))

    #return source image and target image
    def get_image(self):
        ds = pydicom.dcmread(self.img_path)
        img = ds.pixel_array

        img = torch.from_numpy(img).float()
        img = (img - img.min()) / (img.max() - img.min() + 1e-5)
        img = img.unsqueeze(0).unsqueeze(0)

        #flip image if needed
        if self.img_flip:
            img = TF.hflip(img)

        tar_img = img.clone()
        loss_mask = torch.ones_like(img)
        
        #downscale and pad (source)
        new_height = int(self.img_height * self.srcscale)
        new_width = int(self.img_width * self.srcscale)
        img = F.interpolate(img, size = (new_height, new_width), mode = 'bilinear', align_corners = False)

        pad_height = self.img_height - new_height
        pad_width = self.img_width - new_width
        pad_u = pad_height // 2
        pad_d = pad_height - pad_u
        pad_l = pad_width // 2
        pad_r = pad_width - pad_l
        img = F.pad(img, (pad_l, pad_r, pad_u, pad_d), mode = 'constant')

        #crop target image
        _, _, orig_height, orig_width = tar_img.shape
        crop_mask = torch.zeros_like(tar_img)
        crop_mask[:, :, int(orig_height * self.tarcrop):, int(orig_width * self.tarcrop):] = 1
        tar_img = tar_img * crop_mask
        loss_mask = loss_mask * crop_mask

        #downscale and pad (target)
        new_height = int(self.img_height * self.tarscale)
        new_width = int(self.img_width * self.tarscale)
        tar_img = F.interpolate(tar_img, size = (new_height, new_width), mode = 'bilinear', align_corners = False)
        loss_mask = F.interpolate(loss_mask, size = (new_height, new_width), mode = 'bilinear', align_corners = False)

        pad_height = self.img_height - new_height
        pad_width = self.img_width - new_width
        pad_u = pad_height // 2
        pad_d = pad_height - pad_u
        pad_l = pad_width // 2
        pad_r = pad_width - pad_l
        tar_img = F.pad(tar_img, (pad_l, pad_r, pad_u, pad_d), mode = 'constant')
        loss_mask = F.pad(loss_mask, (pad_l, pad_r, pad_u, pad_d), mode = 'constant')

        #rotate image
        img = TF.rotate(img, angle = self.srcrot, interpolation = TF.InterpolationMode.BILINEAR)
        tar_img = TF.rotate(tar_img, angle = self.tarrot, interpolation = TF.InterpolationMode.BILINEAR)
        loss_mask = TF.rotate(loss_mask, angle = self.tarrot, interpolation = TF.InterpolationMode.BILINEAR) 

        #offset image
        def offset_image(img, x, y, fill = 0):
            res = torch.full_like(img, fill)

            stx = max(x, 0)
            enx = min(x + self.img_width, self.img_width)
            sty = max(y, 0)
            eny = min(y + self.img_height, self.img_height)

            res[:, :, sty:eny, stx:enx] = img[:, :, max(0, -y):min(self.img_height - y, self.img_height), max(0, -x):min(self.img_width - x, self.img_width)]

            return res

        img = offset_image(img, self.srcx, self.srcy)
        tar_img = offset_image(tar_img, self.tarx, self.tary)
        loss_mask = offset_image(loss_mask, self.tarx, self.tary)

        return img.squeeze(0), tar_img.squeeze(0), loss_mask.squeeze(0), torch.tensor(self.srcrot - self.tarrot).float().unsqueeze(0), torch.tensor(self.srcscale / self.tarscale).float().unsqueeze(0), torch.tensor(self.srcx - self.tarx).float().unsqueeze(0), torch.tensor(self.srcy - self.tary).float().unsqueeze(0)

In [ ]:
#training datapoints for keypoint detection
keypoint_detection_train_data_points = []
for img_file in train_img:
    for img_flip in range(2):
        for srcrot in np.arange(keypoint_detection_srcrot_st, keypoint_detection_srcrot_en + 1e-8, keypoint_detection_srcrot_int):
            for tarrot in np.arange(keypoint_detection_tarrot_st, keypoint_detection_tarrot_en + 1e-8, keypoint_detection_tarrot_int):
                for tarcrop in np.arange(keypoint_detection_tarcrop_st, keypoint_detection_tarcrop_en + 1e-8, keypoint_detection_tarcrop_int):
                    for srcx in np.arange(keypoint_detection_srcx_st, keypoint_detection_srcx_en + 1e-8, keypoint_detection_srcx_int):
                        for srcy in np.arange(keypoint_detection_srcy_st, keypoint_detection_srcy_en + 1e-8, keypoint_detection_srcy_int):
                            for tarx in np.arange(keypoint_detection_tarx_st, keypoint_detection_tarx_en + 1e-8, keypoint_detection_tarx_int):
                                for tary in np.arange(keypoint_detection_tary_st, keypoint_detection_tary_en + 1e-8, keypoint_detection_tary_int):
                                    keypoint_detection_train_data_points.append(KeypointDetectionDataPoint(img_dir, img_file, img_flip, img_width, img_height, srcrot, tarrot, tarcrop, int(srcx), int(srcy), int(tarx), int(tary)))
print(f'Keypoint detection training set size: {len(keypoint_detection_train_data_points)}')

#validation datapoints for keypoint detection
keypoint_detection_val_data_points = []
for img_file in val_img:
    for img_flip in range(2):
        for srcrot in np.arange(keypoint_detection_srcrot_st, keypoint_detection_srcrot_en + 1e-8, keypoint_detection_srcrot_int):
            for tarrot in np.arange(keypoint_detection_tarrot_st, keypoint_detection_tarrot_en + 1e-8, keypoint_detection_tarrot_int):
                for tarcrop in np.arange(keypoint_detection_tarcrop_st, keypoint_detection_tarcrop_en + 1e-8, keypoint_detection_tarcrop_int):
                    for srcx in np.arange(keypoint_detection_srcx_st, keypoint_detection_srcx_en + 1e-8, keypoint_detection_srcx_int):
                        for srcy in np.arange(keypoint_detection_srcy_st, keypoint_detection_srcy_en + 1e-8, keypoint_detection_srcy_int):
                            for tarx in np.arange(keypoint_detection_tarx_st, keypoint_detection_tarx_en + 1e-8, keypoint_detection_tarx_int):
                                for tary in np.arange(keypoint_detection_tary_st, keypoint_detection_tary_en + 1e-8, keypoint_detection_tary_int):
                                    keypoint_detection_val_data_points.append(KeypointDetectionDataPoint(img_dir, img_file, img_flip, img_width, img_height, srcrot, tarrot, tarcrop, int(srcx), int(srcy), int(tarx), int(tary)))
print(f'Keypoint detection validation set size: {len(keypoint_detection_val_data_points)}')

#testing datapoints for keypoint detection
keypoint_detection_test_data_points = []
for img_file in test_img:
    for img_flip in range(2):
        for srcrot in np.arange(keypoint_detection_srcrot_st, keypoint_detection_srcrot_en + 1e-8, keypoint_detection_srcrot_int):
            for tarrot in np.arange(keypoint_detection_tarrot_st, keypoint_detection_tarrot_en + 1e-8, keypoint_detection_tarrot_int):
                for tarcrop in np.arange(keypoint_detection_tarcrop_st, keypoint_detection_tarcrop_en + 1e-8, keypoint_detection_tarcrop_int):
                    for srcx in np.arange(keypoint_detection_srcx_st, keypoint_detection_srcx_en + 1e-8, keypoint_detection_srcx_int):
                        for srcy in np.arange(keypoint_detection_srcy_st, keypoint_detection_srcy_en + 1e-8, keypoint_detection_srcy_int):
                            for tarx in np.arange(keypoint_detection_tarx_st, keypoint_detection_tarx_en + 1e-8, keypoint_detection_tarx_int):
                                for tary in np.arange(keypoint_detection_tary_st, keypoint_detection_tary_en + 1e-8, keypoint_detection_tary_int):
                                    keypoint_detection_test_data_points.append(KeypointDetectionDataPoint(img_dir, img_file, img_flip, img_width, img_height, srcrot, tarrot, tarcrop, int(srcx), int(srcy), int(tarx), int(tary)))
print(f'Keypoint detection test set size: {len(keypoint_detection_test_data_points)}')

In [ ]:
#training dataset and data loader
keypoint_detection_train_set = ImageDataset(keypoint_detection_train_data_points)
keypoint_detection_train_loader = DataLoader(keypoint_detection_train_set, batch_size = keypoint_detection_train_batch_size, shuffle = True)

#validation dataset and data loader
keypoint_detection_val_set = ImageDataset(keypoint_detection_val_data_points)
keypoint_detection_val_loader = DataLoader(keypoint_detection_val_set, batch_size = keypoint_detection_val_batch_size, shuffle = False)

#testing dataset and data loader
keypoint_detection_test_set = ImageDataset(keypoint_detection_test_data_points)
keypoint_detection_test_loader = DataLoader(keypoint_detection_test_set, batch_size = keypoint_detection_test_batch_size, shuffle = False)

In [ ]:
#quick test
testcnt = 1
testchannelcnt = 1
for imgs1, imgs2, lossmasks, adjrot, adjscale, adjx, adjy in keypoint_detection_train_loader:
    # print(imgs[0].shape)
    # print(ys.shape)
    # print(ys[0])
    # print(aug_scales.shape)
    # print(aug_scales[0])

    for i in range(testchannelcnt):
        plt.imshow(imgs1[0][i], cmap = 'gray')
        plt.axis('off')
        plt.show()

        plt.imshow(imgs2[0][i], cmap = 'gray')
        plt.axis('off')
        plt.show()

        plt.imshow(lossmasks[0][i], cmap = 'gray')
        plt.axis('off')
        plt.show()

        print(adjrot[0])
        print(adjscale[0])
        print(adjx[0])
        print(adjy[0])

    testcnt -= 1
    if testcnt == 0:
        break


## Training

In [ ]:
#model
keypoint_detection_model = KeypointDetectionModel(30, img_height, img_width)

#number of epochs
keypoint_detection_epoch_cnt = 100

#learning rate
keypoint_detection_learning_rate = 1e-3

#early stopping conditions
keypoint_detection_best_val_loss = float('inf')
keypoint_detection_early_stop_lim = 5
keypoint_detection_early_stop_cnt = 0

In [ ]:
#loss function
keypoint_detection_recon_lossfn = nn.MSELoss(reduction = 'sum')
keypoint_detection_dist_lossfn = lambda ypred, yvals : torch.sum(torch.norm(ypred - yvals, dim = 2))

#loss history (in sample)
keypoint_detection_loss_hist = []

In [ ]:
#optimizer
keypoint_detection_optimizer = optim.Adam(keypoint_detection_model.parameters(), lr = keypoint_detection_learning_rate)

In [ ]:
for epoch in range(keypoint_detection_epoch_cnt):
    keypoint_detection_model.to(device)
    keypoint_detection_model.train()

    total_loss = 0

    for source_images, tar_images, loss_masks, adj_rot, adj_scale, adj_x, adj_y in keypoint_detection_train_loader:
        source_images = source_images.to(device)
        tar_images = tar_images.to(device)
        loss_masks = loss_masks.to(device)
        adj_rot = adj_rot.to(device)
        adj_scale = adj_scale.to(device)
        adj_x = adj_x.to(device)
        adj_y = adj_y.to(device)

        model_out, src_keypoints, tar_keypoints = keypoint_detection_model(source_images, tar_images)
        pred_src_keypoints = untransform_keypoints(tar_keypoints, img_height, img_width, adj_rot, adj_scale, adj_x, adj_y)

        loss = keypoint_detection_recon_lossfn(model_out * loss_masks, tar_images * loss_masks)

        keypoint_detection_optimizer.zero_grad()
        loss.backward()
        keypoint_detection_optimizer.step()

        total_loss += loss.item()

    # if epoch % 10 and epoch != epoch_cnt - 1: #only print every x epochs
    #     continue

    print(f'Epoch: {epoch}\n')
    print(f'Loss (in sample): {total_loss / len(keypoint_detection_train_set)}')

    keypoint_detection_loss_hist.append(total_loss / len(keypoint_detection_train_set))

    total_loss = 0

    keypoint_detection_model.eval()
    with torch.no_grad():
        for source_images, tar_images, loss_masks, adj_rot, adj_scale, adj_x, adj_y in keypoint_detection_val_loader:
            source_images = source_images.to(device)
            tar_images = tar_images.to(device)
            loss_masks = loss_masks.to(device)
            adj_rot = adj_rot.to(device)
            adj_scale = adj_scale.to(device)
            adj_x = adj_x.to(device)
            adj_y = adj_y.to(device)

            model_out, src_keypoints, tar_keypoints = keypoint_detection_model(source_images, tar_images)
            pred_src_keypoints = untransform_keypoints(tar_keypoints, img_height, img_width, adj_rot, adj_scale, adj_x, adj_y)

            loss = keypoint_detection_recon_lossfn(model_out * loss_masks, tar_images * loss_masks)
            total_loss += loss.item()

        print(f'Loss (validation): {total_loss / len(keypoint_detection_val_set)} (best {keypoint_detection_best_val_loss / len(keypoint_detection_val_set)})\n')

        print('\n===\n')

    if total_loss > keypoint_detection_best_val_loss:
        keypoint_detection_early_stop_cnt += 1
    else:
        keypoint_detection_early_stop_cnt = 0
        keypoint_detection_best_val_loss = total_loss
        torch.save(keypoint_detection_model.state_dict(), "best_keypoint_detection_model.pth")

    if keypoint_detection_early_stop_cnt >= keypoint_detection_early_stop_lim:
        print('EARLY STOPPED')
        break

plt.plot(keypoint_detection_loss_hist)
plt.show()

In [ ]:
keypoint_detection_model.load_state_dict(torch.load("best_keypoint_detection_model.pth"))

## Validation

In [ ]:
keypoint_detection_model.to(device)
keypoint_detection_model.eval()

total_loss = 0

with torch.no_grad():
    for source_images, tar_images, loss_masks, adj_rot, adj_scale, adj_x, adj_y in keypoint_detection_val_loader:
        source_images = source_images.to(device)
        tar_images = tar_images.to(device)
        loss_masks = loss_masks.to(device)
        adj_rot = adj_rot.to(device)
        adj_scale = adj_scale.to(device)
        adj_x = adj_x.to(device)
        adj_y = adj_y.to(device)

        model_out, src_keypoints, tar_keypoints = keypoint_detection_model(source_images, tar_images)
        pred_src_keypoints = untransform_keypoints(tar_keypoints, img_height, img_width, adj_rot, adj_scale, adj_x, adj_y)

        plt.figure(figsize = (15, 3))

        plt.subplot(1, 5, 1)
        plt.imshow(source_images[0, 0, :, :].cpu().numpy(), cmap = 'gray')

        plt.subplot(1, 5, 2)
        plt.imshow(tar_images[0, 0, :, :].cpu().numpy(), cmap = 'gray')

        plt.subplot(1, 5, 3)
        plt.imshow(model_out[0, 0, :, :].cpu().numpy(), cmap = 'gray')

        plt.subplot(1, 5, 4)
        plt.imshow(source_images[0, 0, :, :].cpu().numpy(), cmap = 'gray')
        plt.scatter(src_keypoints[0, :, 0].cpu().numpy(), src_keypoints[0, :, 1].cpu().numpy(), c = 'b')
        plt.scatter(pred_src_keypoints[0, :, 0].cpu().numpy(), pred_src_keypoints[0, :, 1].cpu().numpy(), c = 'r')

        plt.subplot(1, 5, 5)
        plt.imshow(tar_images[0, 0, :, :].cpu().numpy(), cmap = 'gray')
        plt.scatter(tar_keypoints[0, :, 0].cpu().numpy(), tar_keypoints[0, :, 1].cpu().numpy(), c = 'r')

        plt.show()



## Testing

In [ ]:
keypoint_detection_model.to(device)
keypoint_detection_model.eval()

with torch.no_grad():
    for source_images, tar_images, loss_masks, adj_rot, adj_scale, adj_x, adj_y in keypoint_detection_val_loader:
        source_images = source_images.to(device)
        tar_images = tar_images.to(device)
        loss_masks = loss_masks.to(device)
        adj_rot = adj_rot.to(device)
        adj_scale = adj_scale.to(device)
        adj_x = adj_x.to(device)
        adj_y = adj_y.to(device)

        model_out, src_keypoints, tar_keypoints = keypoint_detection_model(source_images, tar_images)
        pred_src_keypoints = untransform_keypoints(tar_keypoints, img_height, img_width, adj_rot, adj_scale, adj_x, adj_y)

        plt.figure(figsize = (15, 3))

        plt.subplot(1, 5, 1)
        plt.imshow(source_images[0, 0, :, :].cpu().numpy(), cmap = 'gray')

        plt.subplot(1, 5, 2)
        plt.imshow(tar_images[0, 0, :, :].cpu().numpy(), cmap = 'gray')

        plt.subplot(1, 5, 3)
        plt.imshow(model_out[0, 0, :, :].cpu().numpy(), cmap = 'gray')

        plt.subplot(1, 5, 4)
        plt.imshow(source_images[0, 0, :, :].cpu().numpy(), cmap = 'gray')
        plt.scatter(src_keypoints[0, :, 0].cpu().numpy(), src_keypoints[0, :, 1].cpu().numpy(), c = 'b')
        plt.scatter(pred_src_keypoints[0, :, 0].cpu().numpy(), pred_src_keypoints[0, :, 1].cpu().numpy(), c = 'r')

        plt.subplot(1, 5, 5)
        plt.imshow(tar_images[0, 0, :, :].cpu().numpy(), cmap = 'gray')
        plt.scatter(tar_keypoints[0, :, 0].cpu().numpy(), tar_keypoints[0, :, 1].cpu().numpy(), c = 'r')

        plt.show()

## DEBUGGING

In [ ]:
DBG_img, DBG_tar_img, DBG_loss_masks, DBG_adj_rot, DBG_adj_scale, DBG_adj_x, DBG_adj_y = next(iter(keypoint_detection_test_loader))

SANA = DBG_img

plt.imshow(SANA[0, 0], cmap = 'gray')
plt.show()

DBG_img = DBG_img.to(device)

heatmaps, skips = keypoint_detection_model.encoder(DBG_img)

# SANA = skips[3].detach().cpu()
SANA = heatmaps.detach().cpu()

# np.set_printoptions(threshold=np.inf)
# print(SANA[0, 0].numpy())

for i in range(SANA.shape[1]):
    print(SANA[0, i].min())
    print(SANA[0, i].max())

    plt.imshow(SANA[0, i], cmap = 'gray')
    plt.show()

## Other

### Save model

In [ ]:
# torch.save(keypoint_detection_model.state_dict(), "08212025_kp_loss.pth")

### Load model

In [ ]:
keypoint_detection_model.load_state_dict(torch.load("keypoint_model_saves/08242025_kp30withcrop_loss200.pth", map_location = torch.device("cpu")))

### Save encoder only

In [ ]:
# torch.save(keypoint_detection_model.encoder.state_dict(), "08202025_kp30_loss59.pth")

### Testing custom error function

In [ ]:
#######TODO FIXME REMOVE TODO FIXME#######

SANA = KeypointDetectionEncoder(30, img_height, img_width, True)
SANA.load_state_dict(torch.load("keypoint_model_saves/08202025_kp30_loss59_ENCODER.pth", map_location = torch.device("cpu"))) #should be spread out
SANA.to(device)
SANA.eval()

keypoint_detection_model.to(device) #should be default (one point)
keypoint_detection_model.eval()

total_loss = 0

with torch.no_grad():
    for source_images, tar_images, _, adj_rot, adj_scale, adj_x, adj_y in keypoint_detection_val_loader:
        source_images = source_images.to(device)
        tar_images = tar_images.to(device)
        adj_rot = adj_rot.to(device)
        adj_scale = adj_scale.to(device)
        adj_x = adj_x.to(device)
        adj_y = adj_y.to(device)

        test1 = keypoint_detection_model(source_images)
        test2 = SANA(source_images)
        loss1, squares1, area_ratios1 = keypoint_detection_overlap_lossfn(test1, img_height, img_width)
        loss2, squares2, area_ratios2 = keypoint_detection_overlap_lossfn(test2, img_height, img_width)

        plt.figure(figsize = (10, 3))

        plt.subplot(1, 2, 1)
        plt.imshow(squares1[0].detach().cpu(), cmap = 'gray')
        plt.scatter(test1[0, :, 0].detach().cpu(), test1[0, :, 1].detach().cpu(), c = 'b')

        plt.subplot(1, 2, 2)
        plt.imshow(squares2[0].detach().cpu(), cmap = 'gray')
        plt.scatter(test2[0, :, 0].detach().cpu(), test2[0, :, 1].detach().cpu(), c = 'b')
        
        plt.show()

        print(area_ratios1[0].detach().cpu(), loss1[0].detach().cpu())
        print(area_ratios2[0].detach().cpu(), loss2[0].detach().cpu())

In [ ]:
#######TODO FIXME REMOVE TODO FIXME#######

SANA = KeypointDetectionEncoder(30, img_height, img_width, True)
SANA.load_state_dict(torch.load("keypoint_model_saves/08202025_kp30_loss59_ENCODER.pth", map_location = torch.device("cpu"))) #should be spread out
SANA.to(device)
SANA.eval()

keypoint_detection_model.to(device) #should be default (one point)
keypoint_detection_model.eval()

total_loss = 0

with torch.no_grad():
    for source_images, tar_images, _, adj_rot, adj_scale, adj_x, adj_y in keypoint_detection_val_loader:
        source_images = source_images.to(device)
        tar_images = tar_images.to(device)
        adj_rot = adj_rot.to(device)
        adj_scale = adj_scale.to(device)
        adj_x = adj_x.to(device)
        adj_y = adj_y.to(device)

        test1 = keypoint_detection_model(source_images)
        test2 = SANA(source_images)

        loss1, var1 = keypoint_detection_overlap_lossfn(test1)
        loss2, var2 = keypoint_detection_overlap_lossfn(test2)

        plt.figure(figsize = (10, 3))

        plt.subplot(1, 2, 1)
        plt.imshow(source_images[0, 0].detach().cpu(), cmap = 'gray')
        plt.scatter(test1[0, :, 0].detach().cpu(), test1[0, :, 1].detach().cpu(), c = 'b')

        plt.subplot(1, 2, 2)
        plt.imshow(source_images[0, 0].detach().cpu(), cmap = 'gray')
        plt.scatter(test2[0, :, 0].detach().cpu(), test2[0, :, 1].detach().cpu(), c = 'b')
        
        plt.show()

        print(var1.detach().cpu(), loss1.item())
        print(var2.detach().cpu(), loss2.item())